## Imports and paths setup.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from tqdm import tqdm
from ami_utils import ami
from fnn_utils import fnn, plot_fnn
from scipy.signal import butter, filtfilt

base_path = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Exp3_FaceEyeData"

#Where the "Window_XX" subfolders to be created and output CSVs stored:
analysis_path = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/Windowed PCA Analysis"

# Known difficulty conditions (mapped from single-letter codes):
condition_map = {
    "L": "Low",
    "M": "Moderate",
    "H": "High"
}

# Creating windows for the 3-component windowed PCA. Converted to amount of data points for each window by multiplying by number of Hz per second - 60Hz
window_seconds = [10, 20, 30, 60]

window_sizes = [window * 60 for window in window_seconds]
print(window_sizes)

def parse_filename_for_info(filename):
    """
    Extract participant ID and condition from filenames of the form:
        3101_01_L1   (ParticipantID = 3101, Condition = L -> "Low")
        3102_02_M5   (ParticipantID = 3102, Condition = M -> "Moderate")
        ...
    
    Returns (participant_id, condition_string) or (None, None) if not matched.
    Example:
        "3101_01_L1" -> ("3101", "Low")
    """
    basename = os.path.basename(filename)
    # Regex pattern:
    #  - ^(\d{4})   => exactly 4 digits at the start (participant ID)
    #  - .*         => any characters (greedy)
    #  - _(L|M|H)   => underscore followed by L or M or H
    #  - ignoring trailing digits or underscores after the letter
    pattern = r"^(\d{4}).*_(L|M|H)"
    match = re.search(pattern, basename, re.IGNORECASE)
    if match:
        pid = match.group(1)                   # The 4-digit participant ID
        letter = match.group(2).upper()        # "L", "M", or "H"
        cond = condition_map.get(letter, None) # Map to "Low", "Moderate", or "High"
        if cond:
            return pid, cond
    return None, None

[600, 1200, 1800, 3600]


## Defining windowed-PCA function.

In [30]:
def windowed_PCA():
    """
    Perform a windowed PCA analysis for all .csv files in the base directory.
    Outputs reconstructed PC1 and PC2 time-series, loadings, and EVR, separated
    by window size and difficulty condition, with filtering applied to the data.
    """
    # Create output directories with subfolders for each output type
    output_types = ["TimeSeries", "Loadings", "EVR"]
    difficulty_order = ["Low", "Moderate", "High"]
    for w_size in window_sizes:
        for cond in difficulty_order:
            for output_type in output_types:
                path = os.path.join(analysis_path, f"Window_{w_size // 60}", cond, output_type)
                os.makedirs(path, exist_ok=True)

    # Butterworth filter configuration
    b, a = butter(4, 10 / (60 / 2), btype='low')  # 4th order, 10 Hz cutoff, 60 Hz sampling rate

    # Process each file
    for root, _, files in os.walk(base_path):
        for file in files:
            if not file.endswith(".csv"):
                continue

            print(f"Processing file: {file}")
            participant_id, condition = parse_filename_for_info(file)
            if not participant_id or not condition:
                print(f"Skipping file {file}: Unable to extract participant ID or condition.")
                continue

            try:
                # Load time-series data
                file_path = os.path.join(root, file)
                data = pd.read_csv(file_path)
                data = data.dropna()  # Drop missing values if any

                for w_size in window_sizes:
                    print(f"  Window size: {w_size // 60} seconds")
                    step = w_size // 2  # 50% overlap
                    num_samples = len(data)

                    pc1_windows = []  # To store PC1/PC2 for each window
                    evr_data = []  # To store EVR
                    loadings = []  # To store loadings

                    # Perform sliding window PCA
                    for start in range(0, num_samples - w_size + 1, step):
                        end = start + w_size

                        # Debug: Print sliding window indices
                        print(f"    Window start: {start}, end: {end}, size: {end - start}")

                        # Extract data for the window
                        window_data = data.iloc[start:end].copy()

                        # Skip if not enough data points
                        if len(window_data) < w_size:
                            print(f"    Skipping window {start}-{end}: Not enough data points ({len(window_data)} < {w_size})")
                            continue

                        # Centre the window
                        window_data -= window_data.mean(axis=0)

                        # Apply Butterworth filter
                        filtered_data = filtfilt(b, a, window_data, axis=0)

                        # Check lengths after filtering
                        if filtered_data.shape[0] != w_size:
                            print(f"    Skipping window {start}-{end}: Filtered data length mismatch.")
                            continue

                        # Perform PCA
                        pca = PCA(n_components=3)
                        pca.fit(filtered_data)

                        # Reconstruct PC1 and PC2 time-series for this window
                        pc1 = filtered_data @ pca.components_[0]
                        pc2 = filtered_data @ pca.components_[1]

                        # Debug: Check reconstructed shapes
                        print(f"    Reconstructed PC1 shape: {pc1.shape}, PC2 shape: {pc2.shape}")
                        print(f"    Sample indices: {range(start, end)}")

                        # Validate lengths
                        if len(pc1) != w_size or len(pc2) != w_size:
                            print(f"    Skipping window {start}-{end}: PC1/PC2 length mismatch.")
                            continue

                        # Append discrete PC1/PC2 time-series for this window
                        pc1_windows.append(pd.DataFrame({
                            "SampleIndex": range(start, end),
                            "PC1": pc1,
                            "PC2": pc2,
                            "WindowIndex": len(evr_data) + 1
                        }))

                        # Collect EVR and loadings
                        evr = pca.explained_variance_ratio_[:2]
                        evr_data.append({
                            "ParticipantID": participant_id,
                            "WindowIndex": len(evr_data) + 1,
                            "PC1_EVR": evr[0],
                            "PC2_EVR": evr[1]
                        })
                        for i, col in enumerate(data.columns):
                            loadings.append({
                                "ParticipantID": participant_id,
                                "WindowIndex": len(evr_data),
                                "Feature": col,
                                "PC1_Loading": pca.components_[0, i],
                                "PC2_Loading": pca.components_[1, i]
                            })

                    # Save results into respective folders
                    base_dir = os.path.join(analysis_path, f"Window_{w_size // 60}", condition)

                    # Time-Series Output
                    time_series_path = os.path.join(base_dir, "TimeSeries", f"{participant_id}_TimeSeries.csv")
                    if pc1_windows:
                        pd.concat(pc1_windows, ignore_index=False).to_csv(time_series_path, index=False)

                    # EVR Output
                    evr_path = os.path.join(base_dir, "EVR", f"{participant_id}_EVR.csv")
                    pd.DataFrame(evr_data).to_csv(evr_path, index=False)

                    # Loadings Output
                    loadings_path = os.path.join(base_dir, "Loadings", f"{participant_id}_Loadings.csv")
                    pd.DataFrame(loadings).to_csv(loadings_path, index=False)

                print(f"  Successfully processed file: {file}")

            except Exception as e:
                print(f"Error processing file {file}: {e}")

    print("Windowed PCA analysis complete.")


## Define AMI plotting function.

In [37]:
def perform_AMI():
    """
    Perform AMI analysis for all participants across all window sizes and difficulty conditions.
    Save individual plots for participants and aggregate plots for averages.
    """
    
    input_root = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/Windowed PCA Analysis"
    output_root = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/Windowed PCA Analysis/AMI_Plots"
    min_lag = 0
    max_lag = 200

    # Window sizes and difficulty conditions
    window_sizes = ["Window_10", "Window_20", "Window_30", "Window_60"]
    difficulty_conditions = ["Low", "Moderate", "High"]

    # Create output root directory
    os.makedirs(output_root, exist_ok=True)

    for window_size in window_sizes:
        window_path = os.path.join(input_root, window_size)
        for condition in difficulty_conditions:
            condition_path = os.path.join(window_path, condition, "TimeSeries")
            participant_files = [
                os.path.join(condition_path, f)
                for f in os.listdir(condition_path)
                if f.endswith(".csv")
            ]

            # Prepare directory for individual plots
            condition_output_path = os.path.join(output_root, window_size, condition)
            os.makedirs(condition_output_path, exist_ok=True)

            # Store averages for all participants
            avg_amis = []

            for file in participant_files:
                participant_id = os.path.basename(file).split("_")[0]

                # Load time-series data
                data = pd.read_csv(file)

                # Extract unique windows
                window_indices = data['WindowIndex'].unique()

                # Store AMI values for averaging
                all_ami = []

                # Plot each window's AMI in light grey
                for window_index in window_indices:
                    window_data = data[data['WindowIndex'] == window_index]
                    ami_result = ami(window_data['PC1'], min_lag, max_lag)
                    all_ami.append(ami_result[:, 1])  # Store AMI values
                    plt.plot(ami_result[:, 0], ami_result[:, 1], color='lightgrey', alpha=0.7, label='_nolegend_')

                # Calculate and plot average AMI in black
                avg_ami = np.mean(all_ami, axis=0)
                avg_amis.append(avg_ami)  # Store for group average
                plt.plot(ami_result[:, 0], avg_ami, color='black', label='Average AMI')

                # Add labels and legend
                plt.xlabel('Lag')
                plt.ylabel('AMI')
                plt.title(f"AMI Analysis for Participant {participant_id}")
                plt.legend()

                # Save individual plot
                individual_plot_path = os.path.join(condition_output_path, f"{participant_id}_AMI_Plot.png")
                plt.savefig(individual_plot_path)
                plt.close()

            # Calculate and plot the group average (average of averages)
            group_avg_ami = np.mean(avg_amis, axis=0)
            plt.plot(ami_result[:, 0], group_avg_ami, color='blue', label=f'Group Average AMI ({condition})')
            plt.xlabel('Lag')
            plt.ylabel('AMI')
            plt.title(f"Group AMI for {window_size} - {condition}")
            plt.legend()

            # Save group average plot
            group_plot_path = os.path.join(output_root, f"AMI_{window_size.split('_')[1]}_{condition}.png")
            plt.savefig(group_plot_path)
            plt.close()


## Performing EVR Plotting.

In [5]:
# Root directories
input_root = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/Windowed PCA Analysis"
output_root = os.path.join(input_root, "EVR_Plots")  # Output folder for EVR plots
os.makedirs(output_root, exist_ok=True)  # Ensure the output directory exists

# Define conditions
window_sizes = ["Window_10", "Window_20", "Window_30", "Window_60"]
difficulty_conditions = ["Low", "Moderate", "High"]

# Iterate through window sizes and difficulty conditions
for window_size in window_sizes:
    for condition in difficulty_conditions:
        condition_path = os.path.join(input_root, window_size, condition, "EVR")
        participant_files = [
            os.path.join(condition_path, f)
            for f in os.listdir(condition_path)
            if f.endswith(".csv")
        ]

        # Storage for grand averages
        avg_evr_all_participants = []
        
        for file in participant_files:
            participant_id = os.path.basename(file).split("_")[0]
            data = pd.read_csv(file)
            
            # Extract EVR for PC1 and PC2
            avg_evr_pc1 = data["PC1_EVR"].mean()
            avg_evr_pc2 = data["PC2_EVR"].mean()
            avg_evr_all_participants.append([avg_evr_pc1, avg_evr_pc2])
            
            # Plot individual participant EVR
            plt.scatter([1, 2], [avg_evr_pc1, avg_evr_pc2], color='lightgrey', alpha=0.6, label='_nolegend_')
        
        # Compute grand average across all participants
        if avg_evr_all_participants:
            avg_evr_all_participants = np.array(avg_evr_all_participants)
            grand_avg_evr_pc1, grand_avg_evr_pc2 = avg_evr_all_participants.mean(axis=0)
            plt.scatter([1, 2], [grand_avg_evr_pc1, grand_avg_evr_pc2], color='black', label='Group Average EVR')
        
        # Format plot
        plt.xticks([1, 2], labels=["PC1", "PC2"])
        plt.ylabel("Explained Variance Ratio")
        plt.title(f"EVR for {window_size} - {condition}")
        plt.legend()
        
        # Save plot
        plot_filename = f"EVR_{window_size.split('_')[1]}_{condition}.png"
        plt.savefig(os.path.join(output_root, plot_filename))
        plt.close()

## Performing FNN analysis.

In [2]:
# Root directory
input_root = "/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/Windowed PCA Analysis"
output_root_base = os.path.join(input_root, "FNN_Plots")  # Base output folder

# Ensure base output directory exists
os.makedirs(output_root_base, exist_ok=True)

# Define conditions
window_sizes = ["Window_10", "Window_20", "Window_30", "Window_60"]
difficulty_conditions = ["Low", "Moderate", "High"]
tlag_values = [10, 15, 25, 35, 50]  # Time lags to iterate over
min_dim = 1
max_dim = 10

# Loop through each time lag
for tlag in tlag_values:
    output_root = os.path.join(output_root_base, f"lag_{tlag}")  # Create subdirectory for this lag
    os.makedirs(output_root, exist_ok=True)

    for window_size in window_sizes:
        for condition in difficulty_conditions:
            condition_path = os.path.join(input_root, window_size, condition, "TimeSeries")
            
            # Get list of participant files
            participant_files = [
                os.path.join(condition_path, f)
                for f in os.listdir(condition_path)
                if f.endswith(".csv")
            ]

            # Storage for grand averages
            avg_fnn_all_participants = []

            for file in participant_files:
                data = pd.read_csv(file)

                # Extract unique windows
                window_indices = data['WindowIndex'].unique()

                # Store FNN values for averaging
                all_fnn = []

                for window_index in window_indices:
                    window_data = data[data['WindowIndex'] == window_index]
                    fnn_dims, fnn_vals = fnn(window_data['PC1'], tlag, min_dim, max_dim)
                    all_fnn.append(fnn_vals)

                # Calculate average FNN for this participant and store it
                avg_fnn = np.mean(all_fnn, axis=0)
                avg_fnn_all_participants.append(avg_fnn)

            # Compute and plot the grand average across all participants
            if avg_fnn_all_participants:
                group_avg_fnn = np.mean(avg_fnn_all_participants, axis=0)
                plt.plot(fnn_dims, group_avg_fnn, color='blue', label=f'Group Average FNN ({condition})')

                # Format and save the group plot
                plt.xlabel('# Embedding Dimensions')
                plt.ylabel('% False Nearest Neighbors')
                plt.title(f"Group FNN for {window_size} - {condition} (Lag {tlag})")
                plt.xticks(fnn_dims)  # Set x-axis intervals to 1
                plt.legend()
                
                # Save plot
                group_plot_path = os.path.join(output_root, f"FNN_{window_size.split('_')[1]}_{condition}.png")
                plt.savefig(group_plot_path)
                plt.close()

## Running functions.

In [ ]:
#perform_AMI()
#windowed_PCA()
#plot_average_evrs()